# Code GRPO HumanEval - One Click

1. In Colab, select **Runtime > Change runtime type > T4 GPU**.
2. Run the single cell below.

The cell performs the complete smoke-test workflow from a clean Colab runtime. A successful run ends with `SMOKE TEST SUCCESSFUL`.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import files as colab_files

REPOSITORY = 'https://github.com/Hamza-Nadif/code-grpo-humaneval.git'
WORKDIR = Path('/content/code-grpo-humaneval-one-click')
OUTPUT_DIR = Path('outputs/qwen-code-grpo-smoke')

def run(command):
    print('\n$', ' '.join(map(str, command)), flush=True)
    subprocess.run([str(part) for part in command], check=True)

print('STEP 1/6 - Checking the Colab GPU', flush=True)
run(['nvidia-smi'])

print('STEP 2/6 - Loading the latest project version', flush=True)
if WORKDIR.exists():
    run(['git', '-C', WORKDIR, 'pull', '--ff-only', 'origin', 'main'])
else:
    run(['git', 'clone', '--branch', 'main', REPOSITORY, WORKDIR])
os.chdir(WORKDIR)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'Using Git commit: {commit}', flush=True)

print('STEP 3/6 - Installing the reproducible Python stack', flush=True)
run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', 'requirements.txt', '-r', 'requirements-dev.txt',
])

print('STEP 4/6 - Building and validating HumanEval splits', flush=True)
run([sys.executable, 'build_training_data.py', '--output-dir', 'data'])
run([sys.executable, '-m', 'pytest', '-q'])

print('STEP 5/6 - Running the corrected one-step GRPO/QLoRA smoke test', flush=True)
run([
    sys.executable, 'train_grpo.py',
    '--model', 'Qwen/Qwen2.5-Coder-0.5B-Instruct',
    '--train-data', 'data/humaneval_train.jsonl',
    '--eval-data', 'data/humaneval_validation.jsonl',
    '--quantization', '4bit',
    '--precision', 'fp16',
    '--num-generations', '2',
    '--gradient-accumulation-steps', '2',
    '--max-completion-length', '128',
    '--max-steps', '1',
    '--executor', 'local',
    '--allow-local-code-execution',
    '--output-dir', OUTPUT_DIR,
])

print('STEP 6/6 - Verifying and packaging the adapter', flush=True)
required_outputs = [
    OUTPUT_DIR / 'adapter_config.json',
    OUTPUT_DIR / 'adapter_model.safetensors',
    OUTPUT_DIR / 'experiment_config.json',
]
missing = [str(path) for path in required_outputs if not path.is_file()]
if missing:
    raise RuntimeError(f'Training ended without required files: {missing}')
archive = shutil.make_archive('/content/qwen-code-grpo-smoke', 'zip', OUTPUT_DIR)
config = json.loads((OUTPUT_DIR / 'experiment_config.json').read_text())
print('\nSMOKE TEST SUCCESSFUL')
print(f'Git commit: {commit}')
print(f"Precision: {config['configuration']['resolved_precision']}")
print(f'Adapter directory: {WORKDIR / OUTPUT_DIR}')
print(f'Downloadable archive: {archive}')
print('Starting the browser download now...', flush=True)
colab_files.download(archive)
